# K7 Route B GPU engineering acceptance
Bounded fixtures only: 43 and 171 training images, three engineering epochs. No primary launch or production approval. Scientific amendment remains pending. Use the exact matched engineering package.


In [ ]:
import json, hashlib, subprocess, sys, zipfile, os
sys.dont_write_bytecode = True
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
cached = [name for name in sys.modules if name == 'research' or name.startswith('research.')]
assert not cached, 'Cached research imports: ' + repr(cached) + '. Restart the notebook kernel to clear imports; preserve extracted files and attempt records. Do not reload modules from another release.'
from pathlib import Path
from google.colab import drive

UPLOAD = Path('/content/drive/MyDrive/DeepAA_recovery_acceptance_package')
DRIVE_DATA = Path('/content/drive/MyDrive/HAM10000')  # Candidate, verified later.
ENGINEERING_ROOT = Path('/content/drive/MyDrive/DeepAA_recovery_engineering')
ENGINEERING_ID = input('Engineering ID (new test: unique ASCII letters/digits; recovery: original ID): ').strip()
assert 1 <= len(ENGINEERING_ID) <= 64 and ENGINEERING_ID.isascii() and ENGINEERING_ID.isalnum(), 'Use 1-64 ASCII letters/digits'
DURABLE = ENGINEERING_ROOT/ENGINEERING_ID
PAYLOAD = Path('/content/recovery_engineering_payload')
WORKSPACE = Path('/content/recovery_engineering_workspace')
LOCAL_DATA = Path('/content/HAM10000')
ENV = Path('/content/deepaa_env')
PRIMARY_LAUNCH_ENABLED = False
print('Engineering only; primary launch disabled. Dataset not yet verified.')
assert not PAYLOAD.exists() and not WORKSPACE.exists(), 'Extraction/checkout exists. Preserve it; select new absent PAYLOAD and WORKSPACE paths for this corrected release. Never delete attempt records or change DURABLE to bypass a claim.'
drive.mount('/content/drive')


## Verify the transfer before executing packaged code
The senior receipt is the trust anchor. Compare its ZIP SHA-256 with the senior's review output. ZIP extraction rejects unsafe names and verifies every file. No download from the public repository is used.


In [ ]:
receipt = json.loads((UPLOAD/'TRANSFER_RECEIPT.json').read_text())
assert receipt['release'], 'STOP: disabled draft. Senior must commit/review and rebuild release package.'
archive = UPLOAD/receipt['archive']
with archive.open('rb') as f: actual_zip = hashlib.file_digest(f, 'sha256').hexdigest()
assert actual_zip == receipt['sha256'], 'Transfer SHA-256 mismatch'
assert not PAYLOAD.exists(), 'Extraction exists. Do not merge. Review or start a fresh runtime.'
with zipfile.ZipFile(archive) as z:
    names = z.namelist()
    assert len(names) == len(set(names))
    for name in names:
        assert z.getinfo(name).orig_filename == name, 'Unsafe normalized ZIP member'
        p = Path(name)
        assert not p.is_absolute() and '..' not in p.parts and '\\' not in name and ':' not in name
        assert ((z.getinfo(name).external_attr >> 16) & 0o170000) != 0o120000
    z.extractall(PAYLOAD)
package = json.loads((PAYLOAD/'PACKAGE.json').read_text())
def file_sha(p):
    with p.open('rb') as f: return hashlib.file_digest(f, 'sha256').hexdigest()
assert file_sha(PAYLOAD/'PACKAGE.json') == receipt['package_manifest_sha256']
actual = {p.relative_to(PAYLOAD).as_posix(): {'bytes': p.stat().st_size, 'sha256': file_sha(p)} for p in PAYLOAD.rglob('*') if p.is_file() and p != PAYLOAD/'PACKAGE.json'}
assert actual == package['files']
assert package['release'] and package['head'] == receipt['head'] and not package['source_status']
assert package['purpose'] == 'BOUNDED_RECOVERY_ENGINEERING' and package['primary_training_enabled'] is False
assert package['execution_scope'] == dict(K=7, latent_dimension=6, seed=113, mode='bounded recovery engineering', owner='Colab')
# The immutable notebook is inside the verified ZIP; the open copy may autosave.
reviewed_notebook = PAYLOAD/'source_bytes/research/notebooks/recovery_gpu_acceptance.ipynb'
assert file_sha(reviewed_notebook) == receipt['notebook_sha256'], 'Immutable notebook hash mismatch'
def cell_sources(notebook):
    assert isinstance(notebook, dict) and notebook.get('nbformat') == 4
    cells = notebook['cells']
    assert isinstance(cells, list) and cells
    result = []
    for cell in cells:
        assert cell['cell_type'] in ('code', 'markdown'), 'Unexpected cell type'
        source = cell['source']
        assert isinstance(source, str) or (isinstance(source, list) and all(isinstance(line, str) for line in source))
        assert not cell.get('attachments'), 'Unexpected notebook attachments'
        result.append((cell['cell_type'], source if isinstance(source, str) else ''.join(source)))
    return result
from google.colab import _message
live_reply = _message.blocking_request('get_ipynb', timeout_sec=30)
assert isinstance(live_reply, dict) and isinstance(live_reply.get('ipynb'), dict), 'Cannot attest open notebook; stop without a disk-copy fallback'
reviewed_sources = cell_sources(json.loads(reviewed_notebook.read_text()))
assert cell_sources(live_reply['ipynb']) == reviewed_sources, 'Open notebook cell sources/order differ from reviewed package'
print('Open notebook sources verified against immutable packaged notebook; execution metadata/outputs excluded.')
print(json.dumps(receipt, indent=2))
sys.path.insert(0, str(PAYLOAD/'source_bytes'))
from research.code.colab_transfer import restore, run_visible
ROOT = restore(PAYLOAD, WORKSPACE)
print('Restored clean reviewed source with exact packaged bytes and verified D-019 receipt:', ROOT)


## Isolated pinned environment â€” stop on any incompatibility
This downloads Python 3.13.3 and the explicit candidate Linux packages. It does not use Colab's TensorFlow defaults. If a wheel/version or GPU driver is incompatible, **stop and paste the error**; do not downgrade or alter settings. GPU compatibility is not yet established.


In [ ]:
assert not ENV.exists(), 'Environment already exists; do not silently update it.'
run_visible([sys.executable, '-m', 'pip', 'install', 'uv==0.8.22'])
run_visible([sys.executable, '-m', 'uv', 'venv', '--python', '3.13.3', '--seed', str(ENV)])
PYTHON = str(ENV/'bin/python')
INSTALL_REPORT = WORKSPACE/'installation.json'
run_visible([PYTHON, '-m', 'pip', 'install', '--force-reinstall', '--report', str(INSTALL_REPORT), '-r', str(ROOT/'research/requirements_colab.txt')])
run_visible([PYTHON, '-m', 'pip', 'check'])
resolved = json.loads(INSTALL_REPORT.read_text())
lines = []
for item in resolved['install']:
    digest = item['download_info']['archive_info']['hashes']['sha256']
    lines.append(item['metadata']['name']+'=='+item['metadata']['version']+' --hash=sha256:'+digest)
(WORKSPACE/'requirements_resolved_linux.txt').write_text('\n'.join(sorted(lines))+'\n')
print('Target-specific exact wheel-hash lock:', WORKSPACE/'requirements_resolved_linux.txt')


## Verify Drive data and copy to runtime-local storage
Hashes all 10,015 JPEGs and metadata against the Windows-generated source inventory. Frozen split bytes determine membership/order. No dataset upload, split regeneration or Drive-original mutation occurs.


In [ ]:
from research.code.colab_runtime import prepare_images
spec = json.loads((PAYLOAD/'DATASET.json').read_text())
prepare_images(DRIVE_DATA, LOCAL_DATA, spec)
print('Drive and runtime-local image/metadata bytes verified:', len(spec['files']))


## Local orchestration fixtures on the target (no model training)
These tests check preservation, paths, partial-attempt refusal and existing workflow safety. GPU checks run only in the next cell.


In [ ]:
run_visible([PYTHON, '-B', '-m', 'pytest', 'research/tests', '-q', '-p', 'no:cacheprovider'], cwd=ROOT, env=dict(os.environ, PYTHONDONTWRITEBYTECODE='1', MPLBACKEND='Agg'), log=WORKSPACE/'regressions.log')


## Bounded GPU/Drive engineering
This executes three-epoch A/B/C fixtures, real process kills, strict tensor comparisons and verified Drive checkpoints in the engineering namespace. It cannot authorize primary training. Review GPU allocation and Drive quota before execution.


In [ ]:
assert PRIMARY_LAUNCH_ENABLED is False
assert not DURABLE.exists(), 'Preserve earlier engineering evidence; use a new engineering ID.'
run_visible([PYTHON, '-B', '-u', str(ROOT/'research/tools/recovery_engineering.py'), 'run', '--payload', str(PAYLOAD), '--dataset', str(LOCAL_DATA), '--output', str(WORKSPACE/'engineering'), '--durable', str(DURABLE)], env=dict(os.environ, PYTHONHASHSEED='113', TF_DETERMINISTIC_OPS='1', MPLBACKEND='Agg', PYTHONDONTWRITEBYTECODE='1'), log=WORKSPACE/'engineering.log')


## Export setup evidence and stop
Return all results for review. This does not create a readiness or scientific-approval record. Fresh-runtime recovery is a separate operator step described in RECOVERY_ENGINEERING_OPERATOR.txt.


In [ ]:
assert PRIMARY_LAUNCH_ENABLED is False
from research.tools.recovery_engineering import snapshot
SETUP = WORKSPACE/'setup_evidence'
SETUP.mkdir(exist_ok=False)
import shutil
for name in ['RESTORE.json','installation.json','requirements_resolved_linux.txt','regressions.log','engineering.log']:
    shutil.copy2(WORKSPACE/name, SETUP/name)
shutil.copy2(UPLOAD/'TRANSFER_RECEIPT.json', SETUP/'TRANSFER_RECEIPT.json')
snapshot(SETUP, DURABLE/'setup_evidence')
print('Engineering evidence preserved:', DURABLE, 'Primary training remains disabled.')


## No primary execution cell
No launch, automatic next seed, scientific approval or production readiness is generated. Preserve all failures and incomplete transfers. Follow the operator guide for a separate fresh-runtime readback/recovery test.
